# تقييم خطوة الترجمة على المحاضرة (Colab)

يقيس الخط النهائي `final_v1` وخط الأساس (Qwen3-4B، نموذج التشغيل الأول) وباقي الأنظمة المحفوظة **بالبوابة نفسها**:
نفس نوافذ الوقت، ونفس MetricX-24، ونفس فحص الإملاء.

- البيانات تبقى على Google Drive ولا تُرفع إلى أي مكان.
- النتائج تُكتب في `translation_step/evaluations/<eval_id>/auto.json` وتُضاف سطرًا في `evaluations/registry.jsonl`.
- التشغيل: Runtime → Change runtime type → **T4 GPU**، ثم شغّل الخلايا بالترتيب.

In [ ]:
# 1) Drive + code (branch below) + requirements
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

BRANCH = 'evaluation-v2'
RUN_DIR = Path('/content/drive/MyDrive/ArabicDubbingV2/outputs/academic_fresh_pipeline/lesson_bfe775aa22957d44_academic_v2')
assert RUN_DIR.exists(), f'folder not found: {RUN_DIR}'

TR_REPO = Path('/content/translation_step_repo')
if TR_REPO.exists():
    subprocess.run(['git', '-C', str(TR_REPO), 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(TR_REPO), 'checkout', '-q', '-B', BRANCH, 'origin/' + BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    'https://github.com/HamzaAlSamman/AI-Dubbing-Project.git', str(TR_REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu', 'sentencepiece', 'num2words'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'hunspell', 'hunspell-ar'], check=True, capture_output=True)
sys.path.insert(0, str(TR_REPO / 'translation'))
print(subprocess.run(['git', '-C', str(TR_REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# 2) evaluation (5-15 min: MetricX scores only texts it has not scored before)
from dubbing_translation.colab import evaluate_lecture
summary = evaluate_lecture(RUN_DIR, repo_dir=TR_REPO)